# Lab 2 – 在 SQuAD 上進行 SFT 與 LoRA PEFT
**模型：** lab1 pretrain 完的 chekpoint &nbsp;|&nbsp; **資料：** SQuAD v1.1 &nbsp;|&nbsp; **訓練步數：** 各 10

流程：
1. 用類 JSONL 形式觀察 SQuAD 資料  
2. SFT – 10 步（完整模型，answer token 才算 loss）  
3. LoRA PEFT – 10 步（只更新 adapter 權重）

> **SFT/PEFT 不需要 Megatron 前處理。**  
> `make_squad_dataset` 會直接從 HuggingFace 載入 SQuAD 並轉換成訓練格式。


## 0. 環境確認

In [ ]:
import os, sys
from datasets import load_dataset
import json

print("Python :", sys.version.split()[0])
print("GPU    :", end=" ")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import nemo_automodel
print("NeMo Automodel:", nemo_automodel.__version__)

## 1. 檢視 SQuAD 資料格式

SQuAD 會由 `make_squad_dataset` 直接載入，無需手動前處理。
這裡先看幾筆樣本，幫助我們了解模型實際要學什麼。


In [ ]:
squad = load_dataset("rajpurkar/squad", split="train[:5]")

print("SQuAD 欄位：", list(squad.features.keys()))
print("\n" + "─" * 60)
for i, ex in enumerate(squad):
    print(f"\n樣本 {i}")
    print(f"  context  : {ex['context'][:100]!r} …")
    print(f"  question : {ex['question']!r}")
    print(f"  answer   : {ex['answers']['text'][0]!r}")


In [ ]:
# 展示 make_squad_dataset 如何把每一筆樣本整理成類 JSON 紀錄
print("訓練格式（JSONL 風格）：")
print("─" * 60)
for ex in squad:
    context  = ex["context"]
    question = ex["question"]
    answer   = ex["answers"]["text"][0]
    prompt   = f"Context: {context} Question: {question} Answer: "
    record   = {"input": prompt, "output": answer}
    print(json.dumps(record, ensure_ascii=False))
    print()


## 2. SFT – 監督式微調（10 步）

`sft_nemotron3_squad.yaml` 設定的是對 Nemotron-3-Nano-4B 在 SQuAD 上做完整 fine-tuning。
Loss 只在 answer token 上計算（masked cross-entropy）。
合併後的 HF checkpoint 會存到 `checkpoints/sft/`。


In [ ]:
!cat sft_nemotron3_squad.yaml

In [ ]:
!torchrun --nproc-per-node=1 \
    /opt/Automodel/examples/llm_finetune/finetune.py \
    --config sft_nemotron3_squad.yaml \
    --model.pretrained_model_name_or_path /workspace/lab1_pretrain/checkpoints/pretrain/epoch_0_step_9/model/consolidated/

## 3. LoRA PEFT（10 步）

`peft_nemotron3_squad.yaml` 跟 SFT 設定幾乎相同，只是多了一個 `peft:` 區段，
會把 LoRA adapter（`dim=8`、`alpha=32`）注入到所有 linear 層。
只更新約 2M 個 adapter 權重，base model 則被凍結。
合併後的 checkpoint 會存到 `checkpoints/peft/`。


In [ ]:
!diff sft_nemotron3_squad.yaml peft_nemotron3_squad.yaml

In [ ]:
!torchrun --nproc-per-node=1 \
    /opt/Automodel/examples/llm_finetune/finetune.py \
    --config peft_nemotron3_squad.yaml \
    --model.pretrained_model_name_or_path /workspace/lab1_pretrain/checkpoints/pretrain/epoch_0_step_9/model/consolidated/

## 總結

| 方法 | 更新的參數量 | 記憶體 | 適用情境 |
|---|---|---|---|
| SFT | 全部 ~1 B | 高 | 通用 fine-tuning |
| LoRA PEFT | ~2 M adapter | 低 | 資源受限的下游適配 |

兩個方法都用 **masked cross-entropy loss** – 只有 answer token
會對 gradient 有貢獻，context / question 的 prompt 部分不算 loss。


## 4. 釋放磁碟空間

刪除本實驗產生的訓練 checkpoint、lab1 pretrain結束的模型權重。
若還想複習結果可暫時略過此 cell；確認不再使用後再執行。


In [ ]:
import os, shutil
from pathlib import Path

removed = []

# 訓練 checkpoints
shutil.rmtree("checkpoints/")
removed.append("checkpoints/")
shutil.rmtree(Path("/workspace/lab1_pretrain/checkpoints/"))
removed.append("/workspace/lab1_pretrain/checkpoints/")

if removed:
    print("已移除：")
    for r in removed:
        print("  ", r)
else:
    print("沒有可清理的內容")
